# SHIPP — Live DELETE test (Listings **or** Requests)

Proves the real path **Lakebase DELETE → Bronze CDC → Silver** end to end.
Replaces `delete-test-list-001`, `delete-test-listing-002` and `delete-test-request-001`.

## Why this version can be rerun
- The test ID comes from the **`test_id` widget** at the top. Every attempt uses a **new, never-used ID**.
  Gate 1 refuses an ID that already has Bronze history, so an old test can never poison a new one.
- Silver is refreshed **from this notebook** (`dbutils.notebook.run`), so you do not jump between notebooks.
- No gate depends on a variable captured in an earlier run.

## How to run
1. Set the widgets: `entity` = `listing` or `request`; `test_id` = a fresh ID
   (`delete-test-listing-003`, then `-004`, … / `delete-test-request-002`, then `-003`, …).
2. **Run → Clear state and outputs.**
3. Run cells **top to bottom, one gate at a time**. Two gates (3 and 6) ask you to run SQL in the
   **Lakebase SQL editor** — copy the SQL the cell prints, run it there, then continue here.
4. Run cells 1–2 of `validation/01_test_current_state_logic` first if you changed the shared library.

If you abandon a test after Gate 3, run the printed DELETE SQL anyway so no test row stays live in Lakebase.

In [ ]:
%run ../common/shipp_silver_lib

## Gate 1 — Configuration + freshness check

In [ ]:
dbutils.widgets.dropdown("entity", "listing", ["listing", "request"])
dbutils.widgets.text("test_id", "delete-test-listing-003")

ENTITY = dbutils.widgets.get("entity")
TEST_ID = dbutils.widgets.get("test_id").strip()

ENTITIES = {
    "listing": {
        "bronze": "bootcamp_students.shipp_bronze.lb_listings_history",
        "silver": "bootcamp_students.shipp_silver.silver_listings",
        "key": "listing_id",
        "silver_notebook": "../development/10_silver_listings_dev",
        "lakebase_table": "shipp.listings",
        "insert_sql": """SELECT user_id FROM shipp.users WHERE user_id = 'demo-donor-001';

INSERT INTO shipp.listings (
    listing_id, donor_id, title, description, category, condition,
    location, latitude, longitude, available_from, available_until, status
)
VALUES (
    '{test_id}', 'demo-donor-001',
    'DELETE CDC Test Listing',
    'Temporary listing created only to validate CDC delete semantics.',
    'FURNITURE', 'GOOD', 'Abu Dhabi', 24.4976, 54.4075,
    CURRENT_TIMESTAMP, CURRENT_TIMESTAMP + INTERVAL '7 days', 'AVAILABLE'
);

SELECT listing_id, status FROM shipp.listings WHERE listing_id = '{test_id}';""",
    },
    "request": {
        "bronze": "bootcamp_students.shipp_bronze.lb_requests_history",
        "silver": "bootcamp_students.shipp_silver.silver_requests",
        "key": "request_id",
        "silver_notebook": "../development/11_silver_requests_dev",
        "lakebase_table": "shipp.requests",
        "insert_sql": """SELECT user_id FROM shipp.users WHERE user_id = 'demo-requester-001';

INSERT INTO shipp.requests (
    request_id, requester_id, request_text, category, location,
    latitude, longitude, need_by_date, status
)
VALUES (
    '{test_id}', 'demo-requester-001',
    'Temporary request created only to validate CDC delete semantics.',
    'FURNITURE', 'Abu Dhabi', 24.5014, 54.3872,
    CURRENT_TIMESTAMP + INTERVAL '7 days', 'OPEN'
);

SELECT request_id, status FROM shipp.requests WHERE request_id = '{test_id}';""",
    },
}
E = ENTITIES[ENTITY]
KEY = E["key"]

assert TEST_ID.startswith(f"delete-test-{ENTITY}-"), (
    f"test_id must start with 'delete-test-{ENTITY}-' (got '{TEST_ID}'). Fix the widget."
)

bronze_history = spark.table(E["bronze"]).filter(F.col(KEY) == TEST_ID).count()
in_silver = spark.table(E["silver"]).filter(F.col(KEY) == TEST_ID).count()
print(f"ENTITY={ENTITY}  TEST_ID={TEST_ID}")
print("Existing Bronze events for this ID:", bronze_history)
print("Existing Silver rows for this ID:  ", in_silver)

assert bronze_history == 0 and in_silver == 0, (
    f"'{TEST_ID}' has been used before. Change the test_id widget to a new number and rerun Gate 1."
)
print("PASS — fresh test ID.")

## Helpers (Bronze polling + Silver refresh)

In [ ]:
def newest_event(test_id):
    return (
        spark.table(E["bronze"])
        .filter((F.col(KEY) == test_id) & (F.col("_pg_change_type") != "update_preimage"))
        .orderBy(*cdc_newest_first())
        .select("_pg_change_type", "_pg_lsn", "_sort_by", "_timestamp")
        .first()
    )


def wait_for_newest(expected_type, timeout_s=300, every_s=10):
    start = time.time()
    while True:
        ev = newest_event(TEST_ID)
        if ev is not None and ev["_pg_change_type"] == expected_type:
            print(f"Bronze newest event = {expected_type} "
                  f"(lsn={ev['_pg_lsn']}, sort={ev['_sort_by']}) after {time.time() - start:.0f}s")
            return ev
        if time.time() - start > timeout_s:
            raise AssertionError(
                f"STOP: newest Bronze event for {TEST_ID} is "
                f"{ev['_pg_change_type'] if ev else 'missing'}, expected {expected_type} "
                f"after {timeout_s}s. Check the Lakebase SQL ran, then rerun this cell."
            )
        time.sleep(every_s)


def refresh_silver():
    """Run the Silver notebook for this entity. It raises if any of its gates fail."""
    result = dbutils.notebook.run(E["silver_notebook"], 1200)
    print("Silver notebook returned:", result)
    return json.loads(result)


def silver_count(test_id):
    return spark.table(E["silver"]).filter(F.col(KEY) == test_id).count()


print("Helpers ready.")

## Gate 3 — Create the temporary row in Lakebase (manual)

Run the cell, **copy the printed SQL**, run it in the **Lakebase SQL editor**.
Expected there: the parent-user query returns 1 row, the INSERT succeeds, the final SELECT returns 1 row.

In [ ]:
print(E["insert_sql"].format(test_id=TEST_ID))

## Gate 4 — INSERT reached Bronze (polls up to 5 minutes)

In [ ]:
insert_event = wait_for_newest("insert")

## Gate 5 — Refresh Silver; the test row must be present exactly once

Runs the Silver notebook for this entity. If `dbutils.notebook.run` is not allowed in your workspace,
run the Silver notebook by hand (Clear state → Run all) and then rerun only the last two lines.

In [ ]:
refresh_silver()
before_delete = silver_count(TEST_ID)
print("Test row in Silver before DELETE:", before_delete)
assert before_delete == 1, "STOP: test row did not reach Silver before the DELETE."
print("PASS — test row present in Silver before DELETE.")

## Gate 6 — Delete the temporary row in Lakebase (manual)

Copy the printed SQL and run it in the **Lakebase SQL editor**.
Expected: DELETE returns the test ID; the final SELECT returns **0 rows**.

In [ ]:
print(f"""DELETE FROM {E['lakebase_table']}
WHERE {KEY} = '{TEST_ID}'
RETURNING {KEY};

SELECT * FROM {E['lakebase_table']} WHERE {KEY} = '{TEST_ID}';""")

## Gate 7 — DELETE reached Bronze as the newest event (polls up to 5 minutes)

In [ ]:
delete_event = wait_for_newest("delete")
assert (delete_event["_pg_lsn"], delete_event["_sort_by"]) > (insert_event["_pg_lsn"], insert_event["_sort_by"]), (
    "STOP: DELETE is not ordered after the INSERT."
)
print("PASS — Bronze DELETE is newer than the INSERT.")

## Gate 8 — Refresh Silver; the test row must be gone and Silver must match Bronze

The Silver notebook already runs its own reconciliation gate (two independent methods) and
idempotency check; this cell checks the test row specifically and re-runs the reconciliation here.

In [ ]:
refresh_silver()

after_delete = silver_count(TEST_ID)
print("Test row in Silver after DELETE:", after_delete)
assert after_delete == 0, "DELETE VALIDATION FAILED: deleted row still in Silver."

bronze_now = spark.table(E["bronze"])
_, current_now, deleted_now = reconstruct_current_state(bronze_now, KEY)
assert_silver_matches_bronze(E["silver"], bronze_now, current_now, deleted_now, KEY)
print("PASS — deleted row absent; Silver matches Bronze.")

## Gate 9 — Idempotency across a full Silver rerun

In [ ]:
run1 = logical_snapshot(E["silver"], KEY)
refresh_silver()
run2 = logical_snapshot(E["silver"], KEY)

print("Run 1 rows:", len(run1), "| Run 2 rows:", len(run2))
assert run1 == run2, "IDEMPOTENCY FAILED: Silver changed on a rerun with no new Lakebase data."
assert TEST_ID not in {r[KEY] for r in run2}
print(f"PASS — {ENTITY.upper()} DELETE VALIDATED for {TEST_ID}.")

## Evidence to record (README / capstone write-up)

Copy the printed lines from Gates 4, 7, 8 and 9: test ID, Bronze LSNs and `_sort_by` for the INSERT and
DELETE, Silver row counts before/after, and the PASS lines.